In [1]:
import json 
from pathlib import Path
import numpy as np
import pandas as pd

from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn import plotting
from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_destrieux_2009
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

In [2]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [3]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region 
    for region, electrodes in REGION_TO_ELECTRODES.items() 
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

Region-to-Electrode Mapping:
  Middle_Finger (Class 0): ['E1', 'E2', 'E3'] (3 electrodes)
  Hand (Class 1): ['E4', 'E5', 'E6', 'E7'] (4 electrodes)
  Forearm (Class 2): ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'] (6 electrodes)
  Arm (Class 3): ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'] (7 electrodes)


In [ ]:
BIDS_ROOT = Path(r"../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = ["sub-p0002"]
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4


HRF_DELAY = 6.0
WINDOW = 1

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"Subject: {subjects[0]}")
print(f"Runs: {n_runs_per_subject}")
print(f"TR: {tr} s")
print(f"HRF delay: {HRF_DELAY} s")
print(f"Window: {WINDOW} volumes")

RESULTS_DIR = Path("results/4_Classes_ANN")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_LOG = RESULTS_DIR / "4_Classes_ANN_analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')
experiments_results = []

Subject: sub-p0002
Runs: 4
TR: 2.0 s
HRF delay: 6.0 s
Window: 1 volumes


In [5]:
all_events = []
subject = subjects[0]
for run in range(1, n_runs_per_subject + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()

#eletrode -> region mapping
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

print(f"Subject: {subject}")
print(f"Total events: {len(events_df)}")
print(f"Stimulation events: {len(stim_events)}")
print(f"Unique electrodes: {stim_events['trial_type'].nunique()}")
print(f"Unique regions: {stim_events['region'].nunique()}")
print(f"\nSamples per run:")
print(stim_events.groupby('run').size().to_dict())
print(f"\nSamples per electrode:")
electrode_counts = stim_events['trial_type'].value_counts().sort_index()
print(f"  Min: {electrode_counts.min()}, Max: {electrode_counts.max()}, Mean: {electrode_counts.mean():.1f}")
print(f"\nSamples per region:")
region_counts = stim_events['region'].value_counts()
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = region_counts.get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
print(f"\nReference image shape: {first_run_img.shape}")

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
print(f"Number of voxels in original S1 mask: {int(np.sum(mask_data))}")
print(f"Number of voxels in resampled S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

print(f"\nS1 atlas indices: {s1_indices}")
print('Selected atlas regions:')
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

display = plotting.plot_roi(s1_mask_resampled, bg_img=ref_img, 
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()
print(f"\nS1 mask saved to: {FIGURES_DIR / 's1_mask.png'}")

Subject: sub-p0002
Total events: 328
Stimulation events: 160
Unique electrodes: 20
Unique regions: 4

Samples per run:
{1: 40, 2: 40, 3: 40, 4: 40}

Samples per electrode:
  Min: 8, Max: 8, Mean: 8.0

Samples per region:
  Middle_Finger (Class 0): 24 samples
  Hand (Class 1): 32 samples
  Forearm (Class 2): 48 samples
  Arm (Class 3): 56 samples

Reference image shape: (121, 144, 121, 250)
[fetch_atlas_destrieux_2009] Dataset found in C:\Users\duart\nilearn_data\destrieux_2009
Number of voxels in original S1 mask: 1138
Number of voxels in resampled S1 mask: 2187


C:\Users\duart\AppData\Local\Temp\ipykernel_16256\3510676834.py:38: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index          name
    42 L Medial_wall
   117 R Medial_wall

  atlas = fetch_atlas_destrieux_2009()
C:\Users\duart\AppData\Local\Temp\ipykernel_16256\3510676834.py:48: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  masker.fit(first_run_img)



S1 atlas indices: [28]
Selected atlas regions:
  - L G_postcentral

S1 mask saved to: results\4_Classes_ANN_DeepRF\figures\s1_mask.png


c:\Users\duart\miniconda3\envs\somatosensory_mapping\lib\site-packages\numpy\ma\core.py:2820: UserWarning: Warning: converting a masked element to nan.
  _data = np.array(data, dtype=dtype, copy=copy,


In [6]:
X_list = []
y_list = []
run_labels = []

for run in range(1, n_runs_per_subject + 1):
    bold_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...")
    run_events = stim_events[stim_events['run'] == run]
    for indx, (_, event) in enumerate(run_events.iterrows()):
        onset = event["onset"]
        trial_type = event["trial_type"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        
        #Avg of the 3 points, before, during, after
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            vol_data_list = [masker.transform(index_img(bold_img, v)) for v in vol_indices]
            vol_data_list = [vd[0] if len(vd.shape) == 2 else vd for vd in vol_data_list]
            vol_data_avg = np.mean(vol_data_list, axis=0)
            X_list.append(vol_data_avg)
            y_list.append(region)
            run_labels.append(run)

X = np.array(X_list)
y = np.array(y_list)
run_labels = np.array(run_labels)
y_encoded = np.array([REGION_TO_LABEL[region] for region in y])
print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"Number of unique regions: {len(np.unique(y_encoded))}")
print(f"\nLabel distribution:")
for region, label in REGION_TO_LABEL.items():
    count = np.sum(y_encoded == label)
    print(f"  {region} (Class {label}): {count} samples")

Run 1...
Run 2...
Run 3...
Run 4...

Feature matrix shape: (160, 2187)
Labels shape: (160,)
Number of unique regions: 4

Label distribution:
  Middle_Finger (Class 0): 24 samples
  Hand (Class 1): 32 samples
  Forearm (Class 2): 48 samples
  Arm (Class 3): 56 samples


In [7]:
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)

class_weights_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights (imbalanced dataset):")
for label, weight in class_weights_dict.items():
    region_name = LABEL_TO_REGION[label]
    n_samples = np.sum(y_encoded==label)
    print(f"Class {label} ({region_name}): weight={weight:.3f}, n_samples={n_samples}")

unique_labels = sorted(np.unique(y_encoded))
indx_to_label = {indx: LABEL_TO_REGION[indx] for indx in unique_labels}

Class weights (imbalanced dataset):
Class 0 (Middle_Finger): weight=1.667, n_samples=24
Class 1 (Hand): weight=1.250, n_samples=32
Class 2 (Forearm): weight=0.833, n_samples=48
Class 3 (Arm): weight=0.714, n_samples=56


In [8]:
from torch.nn import CrossEntropyLoss, Module, Linear, ReLU, Sequential, Dropout, BatchNorm1d

class SomatotopicANN(Module):
    def __init__(self, input_size, hidden1, hidden2=None, num_classes=4, dropout_rate=0.3):
        super(SomatotopicANN, self).__init__()
        
        layers = []
        
        layers.append(Linear(input_size, hidden1))
        layers.append(BatchNorm1d(hidden1))
        layers.append(ReLU())
        layers.append(Dropout(dropout_rate))
        
        if hidden2 is not None:
            layers.append(Linear(hidden1, hidden2))
            layers.append(BatchNorm1d(hidden2))
            layers.append(ReLU())
            layers.append(Dropout(dropout_rate))
            layers.append(Linear(hidden2, num_classes))
        else:
            layers.append(Linear(hidden1, num_classes))
        
        self.network = Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


In [9]:
single_layer_configs = []
for h1 in [64, 128, 256]:
    for wd in [0.0001, 0.001, 0.01]:
        for dr in [0.2, 0.3, 0.4]:
            single_layer_configs.append({
                'hidden1': h1,
                'hidden2': None,
                'weight_decay': wd,
                'dropout_rate': dr,
                'pca_components': None,
                'architecture': '1-layer'
            })

two_layer_configs = []
layer_combinations = [(256, 64), (128, 32), (64, 16)]
for h1, h2 in layer_combinations:
    for wd in [0.0001, 0.001, 0.01]:
        for dr in [0.2, 0.3, 0.4]:
            two_layer_configs.append({
                'hidden1': h1,
                'hidden2': h2,
                'weight_decay': wd,
                'dropout_rate': dr,
                'pca_components': None,
                'architecture': '2-layer'
            })
 

pca_single_layer_configs = []
for n_comp in [10, 20, 30, 50]:
    for h1 in [16, 8]:
        for wd in [0.2, 0.3, 0.5]:
            for dr in [0.5, 0.6, 0.7]:
                pca_single_layer_configs.append({
                    'hidden1': h1,
                    'hidden2': None,
                    'weight_decay': wd,
                    'dropout_rate': dr,
                    'pca_components': n_comp,
                    'architecture': f'1-layer-PCA{n_comp}'
                })

pca_two_layer_configs = []
for n_comp in [10, 20, 30, 50]:
    for h1, h2 in layer_combinations:
        for wd in [0.2, 0.3, 0.5]:
            for dr in [0.5, 0.6, 0.7]:
                pca_two_layer_configs.append({
                    'hidden1': h1,
                    'hidden2': h2,
                    'weight_decay': wd,
                    'dropout_rate': dr,
                    'pca_components': n_comp,
                    'architecture': f'2-layer-PCA{n_comp}'
                })

all_configs = single_layer_configs + two_layer_configs + pca_single_layer_configs + pca_two_layer_configs

print(f"Total experiments to run: {len(all_configs)}")
print(f"  - Single layer (no PCA): {len(single_layer_configs)}")
print(f"  - Two layer (no PCA): {len(two_layer_configs)}")
print(f"  - Single layer (with PCA): {len(pca_single_layer_configs)}")
print(f"  - Two layer (with PCA): {len(pca_two_layer_configs)}")
print(f"\nFirst 3 configurations:")
for i, config in enumerate(all_configs[:3], 1):
    print(f"  {i}. {config}")

Total experiments to run: 234
  - Single layer (no PCA): 27
  - Two layer (no PCA): 27
  - Single layer (with PCA): 72
  - Two layer (with PCA): 108

First 3 configurations:
  1. {'hidden1': 64, 'hidden2': None, 'weight_decay': 0.0001, 'dropout_rate': 0.2, 'pca_components': None, 'architecture': '1-layer'}
  2. {'hidden1': 64, 'hidden2': None, 'weight_decay': 0.0001, 'dropout_rate': 0.3, 'pca_components': None, 'architecture': '1-layer'}
  3. {'hidden1': 64, 'hidden2': None, 'weight_decay': 0.0001, 'dropout_rate': 0.4, 'pca_components': None, 'architecture': '1-layer'}


In [ ]:
from torch import FloatTensor, LongTensor, manual_seed, no_grad, max, save
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, precision_score, recall_score, f1_score
import torch.optim as optim
import time
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau

def calculate_specificity(y_true, y_pred, class_label):
    y_true_binary = (y_true == class_label).astype(int)
    y_pred_binary = (y_pred == class_label).astype(int)
    tn = np.sum((y_true_binary == 0) & (y_pred_binary == 0))
    fp = np.sum((y_true_binary == 0) & (y_pred_binary == 1))
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("Starting Multi-Model Training...\n")
log_file.write("MULTI-MODEL TRAINING LOG\n\n")

learning_rate = 0.001 #changed lr
target = 60.0
max_no_improvement = 200
batch_size = 32
num_classes = len(unique_labels)
input_size = X.shape[1]

all_experiments_results = []

for exp_idx, config in enumerate(all_configs, 1):
    
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {exp_idx}/{len(all_configs)}")
    print(f"{'='*80}")
    print(f"Architecture: {config['architecture']}")
    print(f"Hidden layers: {config['hidden1']}" + (f" -> {config['hidden2']}" if config['hidden2'] else ""))
    print(f"Weight decay: {config['weight_decay']}, Dropout: {config['dropout_rate']}")
    print(f"{'='*80}\n")
    
    log_file.write(f"\n{'='*80}\n")
    log_file.write(f"EXPERIMENT {exp_idx}/{len(all_configs)}\n")
    log_file.write(f"Config: {config}\n")
    log_file.write(f"{'='*80}\n\n")
    log_file.flush()
    
    hidden1 = config['hidden1']
    hidden2 = config['hidden2']
    weight_decay = config['weight_decay']
    dropout_rate = config['dropout_rate']
    pca_components = config['pca_components']
    
    class_weights_tensor = FloatTensor(class_weights)
    criterion = CrossEntropyLoss(weight=class_weights_tensor)
    
    fold_results = []
    
    start_time = time.time()
    
    for test_run in range(1, n_runs_per_subject + 1):
        print(f"  Fold {test_run}/4...", end=' ')
        
        train_mask = run_labels != test_run
        test_mask = run_labels == test_run
        
        X_train = X[train_mask]
        X_test = X[test_mask]
        y_train = y_encoded[train_mask]
        y_test = y_encoded[test_mask]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        if pca_components is not None:
            pca = PCA(n_components=pca_components, random_state=RANDOM_SEED)
            X_train_processed = pca.fit_transform(X_train_scaled)
            X_test_processed = pca.transform(X_test_scaled)
            current_input_size = pca_components
        else:
            X_train_processed = X_train_scaled
            X_test_processed = X_test_scaled
            current_input_size = input_size
        
        X_train_tensor = FloatTensor(X_train_processed)
        y_train_tensor = LongTensor(y_train)
        X_test_tensor = FloatTensor(X_test_processed)
        y_test_tensor = LongTensor(y_test)
        
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        model = SomatotopicANN(current_input_size, hidden1, hidden2, num_classes, dropout_rate)
        manual_seed(RANDOM_SEED)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=50, min_lr=1e-6)
        
        epoch = 0
        best_test_accuracy = 0.0
        epochs_no_improvement = 0
        best_model_state = None
        
        while True:
            model.train()
            epoch_loss = 0
            correct = 0
            total = 0
            
            for batch_X, batch_y in train_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                _, predicted = max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
            
            train_acc = (correct / total) * 100
            
            model.eval()
            with no_grad():
                test_outputs = model(X_test_tensor)
                _, test_predicted = max(test_outputs.data, 1)
                test_acc = ((test_predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)) * 100
            
            scheduler.step(test_acc)
            if test_acc >= target:
                best_model_state = copy.deepcopy(model.state_dict())
                break
            
            if test_acc > best_test_accuracy:
                best_test_accuracy = test_acc
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improvement = 0
            else:
                epochs_no_improvement += 1
            
            if epochs_no_improvement >= max_no_improvement:
                break
            
            epoch += 1
        
        model.load_state_dict(best_model_state)
        model.eval()
        with no_grad():
            final_outputs = model(X_test_tensor)
            _, final_predictions = max(final_outputs.data, 1)
        
        y_pred = final_predictions.numpy()
        fold_accuracy = accuracy_score(y_test, y_pred)
        
        cm_fold = confusion_matrix(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        specificities = [calculate_specificity(y_test, y_pred, c) for c in range(4)]
        
        fold_results.append({
            'fold': test_run,
            'accuracy': fold_accuracy,
            'epochs': epoch + 1,
            'confusion_matrix': cm_fold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'specificity': np.array(specificities),
            'support': support
        })
        
        print(f"Acc: {fold_accuracy*100:.2f}% ({epoch+1} epochs)")
    
    fold_accuracies = [r['accuracy'] for r in fold_results]
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    total_epochs = sum(r['epochs'] for r in fold_results)
    elapsed_time = time.time() - start_time
    
    cm_total = sum([r['confusion_matrix'] for r in fold_results])
    
    all_precisions = np.array([r['precision'] for r in fold_results])
    all_recalls = np.array([r['recall'] for r in fold_results])
    all_f1s = np.array([r['f1'] for r in fold_results])
    all_specs = np.array([r['specificity'] for r in fold_results])
    
    mean_precision_per_class = np.mean(all_precisions, axis=0)
    mean_recall_per_class = np.mean(all_recalls, axis=0)
    mean_f1_per_class = np.mean(all_f1s, axis=0)
    mean_specificity_per_class = np.mean(all_specs, axis=0)
    
    macro_precision = np.mean(mean_precision_per_class)
    macro_recall = np.mean(mean_recall_per_class)
    macro_f1 = np.mean(mean_f1_per_class)
    macro_specificity = np.mean(mean_specificity_per_class)
    
    total_support = cm_total.sum(axis=1)
    weighted_precision = np.average(mean_precision_per_class, weights=total_support)
    weighted_recall = np.average(mean_recall_per_class, weights=total_support)
    weighted_f1 = np.average(mean_f1_per_class, weights=total_support)
    weighted_specificity = np.average(mean_specificity_per_class, weights=total_support)
    
    print(f"\n  Mean CV Accuracy: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%")
    print(f"  Macro Sensitivity: {macro_recall:.4f}, Macro Specificity: {macro_specificity:.4f}")
    print(f"  Total epochs: {total_epochs}, Time: {elapsed_time:.1f}s")
    
    if pca_components is not None:
        pca_str = f"_pca{pca_components}"
    else:
        pca_str = ""
    
    if hidden2 is not None:
        model_filename = f"model_h{hidden1}-{hidden2}_wd{weight_decay}_dr{dropout_rate}{pca_str}.pt"
    else:
        model_filename = f"model_h{hidden1}_wd{weight_decay}_dr{dropout_rate}{pca_str}.pt"
    
    save(model.state_dict(), RESULTS_DIR / "models"/ model_filename)
    
    cm_filename = model_filename.replace('.pt', '_confusion_matrix.npy')
    np.save(RESULTS_DIR /"models"/ cm_filename, cm_total)
    
    experiment_result = {
        'experiment_id': exp_idx,
        'architecture': config['architecture'],
        'hidden1': hidden1,
        'hidden2': hidden2 if hidden2 is not None else 'None',
        'weight_decay': weight_decay,
        'dropout_rate': dropout_rate,
        'pca_components': pca_components if pca_components is not None else 'None',
        'learning_rate': learning_rate,
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc,
        'macro_sensitivity': macro_recall,
        'macro_specificity': macro_specificity,
        'macro_precision': macro_precision,
        'macro_f1': macro_f1,
        'weighted_sensitivity': weighted_recall,
        'weighted_specificity': weighted_specificity,
        'weighted_precision': weighted_precision,
        'weighted_f1': weighted_f1,
        'per_class_sensitivity': str(mean_recall_per_class.tolist()),
        'per_class_specificity': str(mean_specificity_per_class.tolist()),
        'per_class_precision': str(mean_precision_per_class.tolist()),
        'per_class_f1': str(mean_f1_per_class.tolist()),
        'total_epochs': total_epochs,
        'training_time_s': elapsed_time,
        'model_file': model_filename,
        'confusion_matrix_file': cm_filename
    }
    all_experiments_results.append(experiment_result)
    
    results_df = pd.DataFrame(all_experiments_results)
    results_df.to_csv(RESULTS_DIR / 'all_experiments.csv', index=False)
    
    log_file.write(f"Mean Accuracy: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%\n")
    log_file.write(f"Macro Sensitivity: {macro_recall:.4f}, Macro Specificity: {macro_specificity:.4f}\n")
    log_file.write(f"Model saved: {model_filename}\n")
    log_file.write(f"Confusion matrix saved: {cm_filename}\n\n")
    log_file.flush()

print(f"\n{'='*80}")
print(f"ALL {len(all_configs)} EXPERIMENTS COMPLETE")
print(f"{'='*80}\n")
log_file.write("\nALL EXPERIMENTS COMPLETE\n")

log_file.flush()

Starting Multi-Model Training...


EXPERIMENT 1/234
Architecture: 1-layer
Hidden layers: 64
Weight decay: 0.0001, Dropout: 0.2

  Fold 1/4... Acc: 47.50% (211 epochs)
  Fold 2/4... Acc: 40.00% (213 epochs)
  Fold 3/4... Acc: 37.50% (223 epochs)
  Fold 4/4... Acc: 42.50% (211 epochs)

  Mean CV Accuracy: 41.88% +/- 3.70%
  Macro Sensitivity: 0.4769, Macro Specificity: 0.8039
  Total epochs: 858, Time: 11.7s

EXPERIMENT 2/234
Architecture: 1-layer
Hidden layers: 64
Weight decay: 0.0001, Dropout: 0.3

  Fold 1/4... Acc: 47.50% (206 epochs)
  Fold 2/4... Acc: 37.50% (204 epochs)
  Fold 3/4... Acc: 40.00% (264 epochs)
  Fold 4/4... Acc: 45.00% (209 epochs)

  Mean CV Accuracy: 42.50% +/- 3.95%
  Macro Sensitivity: 0.4661, Macro Specificity: 0.8020
  Total epochs: 883, Time: 10.1s

EXPERIMENT 3/234
Architecture: 1-layer
Hidden layers: 64
Weight decay: 0.0001, Dropout: 0.4

  Fold 1/4... Acc: 30.00% (213 epochs)
  Fold 2/4... Acc: 35.00% (222 epochs)
  Fold 3/4... Acc: 40.00% (216 epochs)
  F

In [11]:
log_file.close()
print(f"\n{'='*70}")
print(f"All experiments complete! Results saved to: {RESULTS_DIR}")
print(f"{'='*70}")
print(f"\nOutput files:")
print(f"  - Log: {OUTPUT_LOG}")
print(f"  - All experiments: {RESULTS_DIR / 'all_experiments.csv'}")
print(f"  - Models: {RESULTS_DIR / '*.pt'}")


All experiments complete! Results saved to: results\4_Classes_ANN_DeepRF

Output files:
  - Log: results\4_Classes_ANN_DeepRF\4_Classes_ANN_DeepRF_analysis_log.txt
  - All experiments: results\4_Classes_ANN_DeepRF\all_experiments.csv
  - Models: results\4_Classes_ANN_DeepRF\*.pt
